In [10]:
import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
import os, re, random
import numpy as np
import pickle
from tqdm.auto import tqdm
from IPython.display import clear_output
from IPython.core.interactiveshell import InteractiveShell

from sklearn.model_selection import train_test_split
InteractiveShell.ast_node_interactivity = "all"

print(torch.__version__)
print(torchaudio.__version__)

2.2.0
2.2.0


# Getting all wav files

In [14]:
path = '/home/iblitvinov@vniia.int/projects/stock_data/'
sound_list = list()

for root, dirs, files in os.walk(path, topdown=False):
    for file in files:
        if re.search(".wav", file) and re.search("user", root):
            file_path = os.path.join(root, file)
            sound_list.append(file_path)

random.shuffle(sound_list)
len(sound_list)

3625

In [12]:
X_train, X_valid = train_test_split(sound_list, test_size=0.15, random_state=42, stratify=y)

len(X_train), len(X_valid)

(3081, 544)

## For deleting prev results

In [13]:
for sound in sound_list:
    if re.search("new", sound):
        os.remove(sound)

In [15]:
import math

import IPython.display as ipd
import matplotlib.pyplot as plt

from torchaudio.utils import download_asset

# Experiments

In [16]:
def create_noise(SNR_db, p, noise_size):
    snr = 10.0 ** (SNR_db / 10.0)
    sample_noise = np.random.normal(0, np.sqrt(p / snr), noise_size)
    return sample_noise

def noisy_audio(speech, snr=3):
    noise = create_noise(snr, torch.var(speech[0]), len(speech[0]))
    noisy_speech = speech + noise
    return noisy_speech.to(torch.float32)

# Creating new dataset

In [17]:
effects = {
    'Reverb': [[["channels", '1'], ['reverb', '-w'], ["rate", '16000'], ['speed', str(rate)]] for rate in [0.9, 1, 1.1]],
    'Lowpass': [[["channels", '1'], ["lowpass", "-1", "300"], ["rate", '16000'], ['speed', str(rate)]] for rate in [0.9, 1, 1.1]],
    'Speed_pitch': [[["channels", '1'], ["rate", '16000'], ['pitch', f'{level}']] for level in [-250, -200, -150, 150, 200, 250]],
    'Contrast': [[["channels", '1'], ["rate", '16000'], ['flanger', '10', '10', '80'], ['contrast', str(val)], ['pitch', f'{level}']] for val in [10, 25, 75] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Fir': [[["channels", '1'], ["rate", '16000'], ['flanger', '12', '10', '80'], ['fir', str(val), str(val), str(val)]] for val in [0.5, 1, 2]],
    'Hilbert': [[["channels", '1'], ["rate", '16000'], ['hilbert'], ['overdrive'], ['pitch', f'{level}']] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Phaser': [[["channels", '1'], ["rate", '16000'], ['vad'], ['phaser', '0.8', '0.6', '5', '0.24', '2', '-s'], ['pitch', f'{level}']] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Stretch': [[["channels", '1'], ['stretch', str(val)], ['pitch', f'{level}']] for val in [0.6, 0.8] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Chorus': [[['channels', '1'], ['chorus', '0.7', '0.9', '55', '0.4', '0.25', '2', val], ['pitch', f'{level}']] for val in ['-t', '-s'] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Noise_Stretch': [[["channels", '1'], ['stretch', str(val)], ['pitch', f'{level}']] for val in [0.6, 0.8] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Noise_Chorus': [[['channels', '1'], ['chorus', '0.7', '0.9', '55', '0.4', '0.25', '2', val], ['pitch', f'{level}']] for val in ['-t', '-s'] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Noise_speed_pitch': [[["channels", '1'], ["rate", '16000'], ['pitch', f'{level}']] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Noise_Loudness': [[["channels", '1'], ["rate", '16000'], ['flanger', '12', '10', '80'], ['loudness', str(val)], ['pitch', f'{level}']] for val in [1, 10] for level in [-250, -200, -150, 0, 150, 200, 250]],
    'Noise_delay': [[["channels", '1'], ["rate", '16000'], ['fir', '0.25', '0.25', '0.25'], ['delay', str(val)]] for val in [0.3, 0.5, 0.7, 0.9]],
    'Noise_non_effects': [[["channels", '1'], ["rate", '16000']]]
}

In [53]:
def create_outputdata(sound_list, isTrain=False):
    data = []
# RandAugment
    while tqdm(sound_list):
        clear_output(wait=True)
        curr_audio_path = sound_list.pop()
        root = curr_audio_path.split('/')[-2]
        audio = curr_audio_path.split('/')[-1].split('.')[0]

        file_name = os.path.join(root, f'{audio}.wav')
        label = audio.split('_')[0]
        temp = {'name': file_name, 'label': label}
        data.append(temp)

        if isTrain:
            type_effect = random.choice(list(effects.keys()))
            if type_effect != 'Non_effects':
                for i, effect in enumerate(effects[type_effect]):
                    new_wave, sr = torchaudio.sox_effects.apply_effects_file(curr_audio_path, effect, channels_first=True)
                    
                    if type_effect.split('_')[0] == 'Noise':
                        new_wave = noisy_audio(new_wave, snr=random.choice([1, 3]))
                        
                    file_name = os.path.join(root, f'{audio}_new_{i+1}.wav')
                    temp = {'name': file_name, 'label': label}
                    torchaudio.save(os.path.join(path, file_name), new_wave, sr)
                    data.append(temp)
                    
    print( 'Output data has been created\n', len(data), data )
    return data

0it [00:00, ?it/s]

In [61]:
import torchaudio

def check_data(data, isTrain=False):
    for sound in tqdm(data):
        data_path = os.path.join(path, sound['name'])
        try:
            audio = torchaudio.load(data_path)
            if audio[0].size(1) == 0:
                print(audio[0].size(1))
                data.remove(sound)
                os.remove(data_path)
                print(f'Removed {sound["name"]}')
        except RuntimeError:
            print(sound)
            data.remove(sound)
    
    return data

train_data = create_outputdata(X_train, isTrain=True)
valid_data = create_outputdata(X_train, isTrain=False)



  0%|          | 0/32851 [00:00<?, ?it/s]

32851


33260

In [ ]:
train_data = check_data(train_data)
valid_data = check_data(valid_data)

In [ ]:
train_name = 'train_data_base_audio.pickle'
valid_name = 'valid_data_base_audio.pickle'

with open(f'{path}/{train_name}', 'wb') as f:
    pickle.dump(train_data, f)
    print( len(train_data) )

with open(f'{path}/{valid_name}', 'wb') as f:
    pickle.dump(valid_data, f)
    print( len(valid_data) )